# Task 3, Step 3: Off-Policy Distillation (Teacher-Generated Prefixes)

1. **Frozen teacher-rollout pool:** walk the shared seeded round-robin schedule over the 800 training prompts
   (500 GSM8K train + 300 short-form train; math chosen whenever it is further behind its 30,000-token quota than
   short-form is behind its 10,000-token quota) and sample one teacher response per presentation (temperature 1,
   top-p 1, caps 192 / 32). The pool is generated once (seed 6304) and its generation time is part of the cost.
2. **Training (seeds 6304, 6305):** LoRA r=8 / alpha=16 on q, k, v, o of the frozen FP16 0.5B student; forward
   KL(p_teacher || q_student) over the full vocabulary at every response position (soft targets, never the sampled
   token as a label); updates per >= 256 teacher-scored tokens with KL normalised by the update's token count.
No GSM8K test question or held-out short-form prompt enters the pool (checked in notebook 1).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from shared.src.io import read_jsonl, load_json, save_json
from shared.src.plotting import save_figure
from task3_speculative.src.distill import load_models, build_teacher_pool, train

M = load_models(load_draft=False)
tok, target = M["tok"], M["target"]
RUNS = ctx.checkpoints / "distill"
pool = build_teacher_pool(target, tok, RUNS / "teacher_pool.jsonl.gz")
rows = pool["rows"]
pool_stats = dict(n_responses=len(rows), teacher_generation_s=pool["time_s"],
                  tokens={d: sum(r["n_tokens"] for r in rows if r["domain"] == d) for d in ("math", "short")},
                  eos_frac=sum(r["eos"] for r in rows) / len(rows), cap_truncated=sum(r["truncated"] for r in rows))
save_json(ctx.results / "distill" / "teacher_pool_stats.json", pool_stats)
pool_stats

In [ ]:
summaries = [train("off", seed, target, tok, RUNS, pool=rows) for seed in (6304, 6305)]
pd.DataFrame([{k: v for k, v in s.items() if k not in ("lora", "opt", "quota", "caps")} for s in summaries])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for s in (6304, 6305):
    u = pd.DataFrame(read_jsonl(RUNS / f"off_policy_s{s}" / "updates.jsonl"))
    u["scored"] = u.scored_math + u.scored_short
    ax[0].plot(u.scored, u.kl, label=f"off-policy s{s}")
    ax[1].plot(u.update, u.tokens, label=f"s{s}")
ax[0].set_xlabel("teacher-scored response tokens"); ax[0].set_ylabel("KL(p_T || q) per token (update mean)"); ax[0].legend()
ax[1].set_xlabel("optimizer update"); ax[1].set_ylabel("tokens in update")
save_figure(ctx.figures, "offpolicy_training_curves")
for s in (6304, 6305):
    save_json(ctx.results / "distill" / f"off_policy_s{s}_summary.json", load_json(RUNS / f"off_policy_s{s}" / "summary.json"))